In [1]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(0)

In [2]:
d = 20

lambda_min = 1
lambda_max = 100
lambdas = np.linspace(lambda_min, lambda_max, d)

H = np.diag(lambdas)

Define quadratic and gradient

In [3]:
def f(w):
    return 0.5 * w.T @ H @ w

def grad(w):
    return H @ w

Heavy Ball (Polyak) Update

In [4]:
def heavy_ball(w0, eta, beta, steps):
    w = w0.copy()
    v = np.zeros_like(w)
    history = []

    for _ in range(steps):
        v = beta * v + grad(w)
        w = w - eta * v
        history.append(f(w))

    return np.array(history)

In [5]:
kappa = lambda_max / lambda_min

eta_opt = 4 / ((np.sqrt(lambda_max) + np.sqrt(lambda_min))**2)

beta_opt = ((np.sqrt(lambda_max) - np.sqrt(lambda_min)) /
            (np.sqrt(lambda_max) + np.sqrt(lambda_min)))**2

print("eta_opt:", eta_opt)
print("beta_opt:", beta_opt)
print("theoretical spectral radius:", np.sqrt(beta_opt))

eta_opt: 0.03305785123966942
beta_opt: 0.6694214876033059
theoretical spectral radius: 0.8181818181818182


In [6]:
w0 = np.random.randn(d)

hb_hist = heavy_ball(w0, eta_opt, beta_opt, 200)

In [7]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=hb_hist,
        mode='lines',
        name='Heavy Ball',
    )
)

fig.update_layout(
    title="Heavy Ball Convergence (Interactive)",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",   # logarithmic scale
)

fig.show()

Compare with Gradient Descent

In [8]:
def gradient_descent(w0, eta, steps):
    w = w0.copy()
    history = []

    for _ in range(steps):
        w = w - eta * grad(w)
        history.append(f(w))

    return np.array(history)

eta_gd = 2 / (lambda_min + lambda_max)

gd_hist = gradient_descent(w0, eta_gd, 200)

In [9]:
fig = go.Figure()

fig.add_trace(go.Scatter(y=gd_hist, mode='lines', name='Gradient Descent'))
fig.add_trace(go.Scatter(y=hb_hist, mode='lines', name='Heavy Ball'))

fig.update_layout(
    title="GD vs Heavy Ball (Interactive)",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",
)

fig.show()

In [10]:
beta_bad = 0.9
hb_bad = heavy_ball(w0, eta_opt, beta_bad, 200)

fig = go.Figure()
fig.add_trace(go.Scatter(y=hb_bad, mode='lines', name='Unstable HB'))

fig.update_layout(
    title="Heavy Ball with Wrong Beta",
    yaxis_type="log",
)

fig.show()

### 2D case for animation


In [11]:
d2 = 2
lambdas_2d = np.array([lambda_min, lambda_max])
H2 = np.diag(lambdas_2d)

def f2(w):
    return 0.5 * w.T @ H2 @ w

def grad2(w):
    return H2 @ w

In [12]:
def heavy_ball_trajectory(w0, eta, beta, steps):
    w = w0.copy()
    v = np.zeros_like(w)
    trajectory = [w.copy()]

    for _ in range(steps):
        v = beta * v + grad2(w)
        w = w - eta * v
        trajectory.append(w.copy())

    return np.array(trajectory)

In [13]:
w0_2d = np.array([5.0, 5.0])
traj = heavy_ball_trajectory(w0_2d, eta_opt, beta_opt, 60)

In [14]:

fig = go.Figure()

# Add contour of quadratic
x = np.linspace(-6, 6, 200)
y = np.linspace(-6, 6, 200)
X, Y = np.meshgrid(x, y)
Z = 0.5 * (lambdas_2d[0] * X**2 + lambdas_2d[1] * Y**2)

fig.add_trace(
    go.Contour(
        x=x,
        y=y,
        z=Z,
        colorscale="Viridis",
        showscale=False,
        opacity=0.5
    )
)

# Add animated trajectory
fig.add_trace(
    go.Scatter(
        x=traj[:,0],
        y=traj[:,1],
        mode="lines+markers",
        line=dict(color="red"),
        name="Heavy Ball Path"
    )
)

fig.update_layout(
    title="Heavy Ball Spiral Convergence (2D)",
    xaxis_title="w1",
    yaxis_title="w2",
    width=700,
    height=700
)

fig.show()

## Convex NAG — O(1/k²)

Convex Setup (L-smooth quadratic)

In [15]:
# Use same H from before (diagonal Hessian)
L = lambda_max   # Lipschitz constant of gradient
eta_nag = 1 / L

def f(w):
    return 0.5 * w.T @ H @ w

def grad(w):
    return H @ w

Convex NAG Update

In [16]:
def nag_convex(w0, steps):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for k in range(1, steps+1):
        beta_k = (k - 1) / (k + 2)
        y = w + beta_k * (w - w_prev)
        w_next = y - eta_nag * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)

In [17]:
w0 = np.random.randn(d)

nag_hist = nag_convex(w0, 2000)

In [18]:
k_vals = np.arange(1, len(nag_hist)+1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_hist,
    mode='lines',
    name='NAG'
))

fig.add_trace(go.Scatter(
    x=k_vals,
    y=1/(k_vals**2),
    mode='lines',
    name='1/k^2 reference',
    line=dict(dash='dash')
))

fig.update_layout(
    title="Convex NAG Convergence (O(1/k^2))",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",
)

fig.show()

What if : eta_nag = 1.1 / L

In [19]:
eta_nag = 1.1 / L   # intentionally larger than 1/L

def nag_convex(w0, steps):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for k in range(1, steps+1):
        beta_k = (k - 1) / (k + 2)
        y = w + beta_k * (w - w_prev)
        w_next = y - eta_nag * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)



w0 = np.random.randn(d)

nag_hist = nag_convex(w0, 2000)


k_vals = np.arange(1, len(nag_hist)+1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_hist,
    mode='lines',
    name='NAG (eta = 1.1/L)'
))

fig.update_layout(
    title="Convex NAG with Too Large Step Size",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log"
    )

fig.show()



In [20]:
# --- Stable case ---
eta_stable = 1 / L

def nag_with_eta(w0, steps, eta):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for k in range(1, steps+1):
        beta_k = (k - 1) / (k + 2)
        y = w + beta_k * (w - w_prev)
        w_next = y - eta * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)


w0 = np.random.randn(d)

nag_stable = nag_with_eta(w0,2000 , eta_stable)
nag_unstable = nag_with_eta(w0, 2000, 1.1/L)


# --- Interactive Plot ---
import plotly.graph_objects as go

k_vals = np.arange(1, 2001)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_stable,
    mode='lines',
    name='NAG (eta = 1/L)',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_unstable,
    mode='lines',
    name='NAG (eta = 1.1/L)',
    line=dict(color='red')
))

fig.update_layout(
    title="Convex NAG: Stability Boundary Comparison",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",
)

fig.show()

In [21]:
# --- Stable case ---
eta_stable = 1 / L

def nag_with_eta(w0, steps, eta):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for k in range(1, steps+1):
        beta_k = (k - 1) / (k + 2)
        y = w + beta_k * (w - w_prev)
        w_next = y - eta * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)


w0 = np.random.randn(d)

nag_stable = nag_with_eta(w0,2000 , eta_stable)
nag_unstable = nag_with_eta(w0, 2000, 1.4/L)


# --- Interactive Plot ---
import plotly.graph_objects as go

k_vals = np.arange(1, 2001)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_stable,
    mode='lines',
    name='NAG (eta = 1/L)',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_unstable,
    mode='lines',
    name='NAG (eta = 1.4/L)',
    line=dict(color='red')
))

fig.update_layout(
    title="Convex NAG: Stability Boundary Comparison",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",
)

fig.show()

## Strongly Convex NAG

In [22]:
mu = lambda_min
L = lambda_max
kappa = L / mu

eta_sc = 1 / L

beta_sc = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)

print("eta:", eta_sc)
print("beta:", beta_sc)
print("theoretical rate:", 1 - np.sqrt(mu/L))

eta: 0.01
beta: 0.8181818181818182
theoretical rate: 0.9


In [23]:
def nag_strong(w0, steps):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for _ in range(steps):
        y = w + beta_sc * (w - w_prev)
        w_next = y - eta_sc * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)

In [24]:
w0 = np.random.randn(d)

nag_sc_hist = nag_strong(w0, 200)

Compare with GD

In [25]:
def gradient_descent(w0, steps):
    w = w0.copy()
    history = []

    for _ in range(steps):
        w = w - (1/L) * grad(w)
        history.append(f(w))

    return np.array(history)

gd_hist = gradient_descent(w0, 200)

In [26]:
k_vals = np.arange(1, 201)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=gd_hist,
    mode='lines',
    name='Gradient Descent',
    line=dict(color='orange')
))

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_sc_hist,
    mode='lines',
    name='Strongly Convex NAG',
    line=dict(color='blue')
))

fig.update_layout(
    title="Strongly Convex: GD vs NAG",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log"
)

fig.show()

Wrong μ Estimate (Mis-tuned β)

In [27]:
# Mis-specified kappa (pretend conditioning worse than reality)
kappa_wrong = 4 * kappa  # exaggerate condition number

beta_wrong = (np.sqrt(kappa_wrong) - 1) / (np.sqrt(kappa_wrong) + 1)

print("Correct beta:", beta_sc)
print("Wrong beta:", beta_wrong)


def nag_strong_wrong_beta(w0, steps):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for _ in range(steps):
        y = w + beta_wrong * (w - w_prev)
        w_next = y - eta_sc * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)


nag_wrong_hist = nag_strong_wrong_beta(w0, 200)

Correct beta: 0.8181818181818182
Wrong beta: 0.9047619047619048


In [28]:
fig = go.Figure()

fig.add_trace(go.Scatter(y=nag_sc_hist, mode='lines',
                         name='Correct NAG', line=dict(color='blue')))

fig.add_trace(go.Scatter(y=nag_wrong_hist, mode='lines',
                         name='Wrong Beta', line=dict(color='red')))

fig.update_layout(
    title="Strongly Convex NAG: Beta Mis-specification",
    yaxis_type="log",
)

fig.show()

Curvature Drift (Simulate Changing Hessian)

In [29]:
def nag_with_drift(w0, steps):
    w_prev = w0.copy()
    w = w0.copy()
    history = []

    for k in range(steps):

        # Change curvature after 100 iterations
        if k == 100:
            global H
            H = np.diag(np.linspace(lambda_min, 4*lambda_max, d))  # increase L

        y = w + beta_sc * (w - w_prev)
        w_next = y - eta_sc * grad(y)

        w_prev = w
        w = w_next

        history.append(f(w))

    return np.array(history)


nag_drift_hist = nag_with_drift(w0, 200)

In [30]:
nag_sc_hist
nag_drift_hist

array([1.43456837e+02, 2.66985038e+01, 1.66035350e+01, 1.33672170e+01,
       8.82666410e+00, 5.75685684e+00, 3.90189311e+00, 2.63576383e+00,
       1.88287315e+00, 1.53277826e+00, 1.33516281e+00, 1.13560317e+00,
       9.29174007e-01, 7.56591031e-01, 6.30194123e-01, 5.34886689e-01,
       4.53860823e-01, 3.80608869e-01, 3.15847232e-01, 2.61389860e-01,
       2.17427725e-01, 1.82717686e-01, 1.55452582e-01, 1.33781070e-01,
       1.16026795e-01, 1.00840413e-01, 8.73185812e-02, 7.50177710e-02,
       6.38430094e-02, 5.38725414e-02, 4.52002143e-02, 3.78436373e-02,
       3.17216765e-02, 2.66783349e-02, 2.25247033e-02, 1.90775305e-02,
       1.61834368e-02, 1.37270456e-02, 1.16272243e-02, 9.82771836e-03,
       8.28749780e-03, 6.97370065e-03, 5.85771936e-03, 4.91362187e-03,
       4.11778232e-03, 3.44889599e-03, 2.88799130e-03, 2.41835656e-03,
       2.02541872e-03, 1.69661340e-03, 1.42125197e-03, 1.19037111e-03,
       9.96551846e-04, 8.33708802e-04, 6.96864669e-04, 5.81931098e-04,
      

In [31]:
k_vals = np.arange(1, len(nag_sc_hist)+1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_sc_hist,
    mode='lines',
    name='Strongly Convex NAG (Stable)',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=k_vals,
    y=nag_drift_hist,
    mode='lines',
    name='NAG with Curvature Drift',
    line=dict(color='red')
))

# Vertical line showing where curvature changed
fig.add_vline(
    x=100,
    line_width=2,
    line_dash="dash",
    line_color="green"
)

fig.update_layout(
    title="Strongly Convex NAG: Curvature Drift Instability",
    xaxis_title="Iteration",
    yaxis_title="f(w)",
    yaxis_type="log",
)

fig.show()

# Acceleration Methods: Final Comparative Conclusions

---

# 1. Polyak Heavy Ball (Momentum)

## Model

$$
v_{k+1} = \beta v_k + \nabla f(w_k)
$$

$$
w_{k+1} = w_k - \eta v_{k+1}
$$

Equivalent second-order recurrence:

$$
w_{k+1}
=
w_k - \eta \nabla f(w_k)
+ \beta (w_k - w_{k-1})
$$

---

## Quadratic Case

Assume:

$$
f(w) = \frac12 w^T H w
$$

with condition number:

$$
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

Optimal tuning:

$$
\eta^* = \frac{4}{(\sqrt{\lambda_{\max}} + \sqrt{\lambda_{\min}})^2}
$$

$$
\beta^* =
\left(
\frac{\sqrt{\kappa} - 1}
{\sqrt{\kappa} + 1}
\right)^2
$$

Contraction factor:

$$
\rho =
\frac{\sqrt{\kappa} - 1}
{\sqrt{\kappa} + 1}
$$

---

## Strengths

- $\sqrt{\kappa}$ acceleration on quadratics  
- Simple implementation  
- Exponential moving average behavior  
- Reduces high-frequency noise  

---

## Weaknesses

- No $O(1/k^2)$ guarantee for general convex functions  
- No global Lyapunov proof  
- Requires spectrum knowledge for optimal tuning  
- Sensitive to $\beta$ mis-specification  
- Fragile under curvature drift  

---

# 2. Convex Nesterov Accelerated Gradient (NAG)

## Model

Time-varying momentum:

$$
\beta_k = \frac{k-1}{k+2}
$$

Look-ahead step:

$$
y_k = w_k + \beta_k (w_k - w_{k-1})
$$

Gradient update:

$$
w_{k+1} = y_k - \frac{1}{L} \nabla f(y_k)
$$

---

## Convergence Rate

For $L$-smooth convex functions:

$$
f(w_k) - f^*
\le
\frac{C}{k^2}
$$

This matches the optimal lower bound for first-order methods.

---

## Interpretation

Continuous-time limit:

$$
\ddot{x} + \frac{3}{t} \dot{x} + \nabla f(x) = 0
$$

Damping decreases over time.

Early phase:
- High damping
- Stable

Late phase:
- Low damping
- Accelerated motion

---

## Strengths

- Provable $O(1/k^2)$ rate  
- Constructible Lyapunov function  
- Optimal for smooth convex optimization  

---

## Weaknesses

- Requires $\eta \le 1/L$  
- Sensitive to step size  
- Oscillatory behavior  
- No linear convergence  

---

# 3. Strongly Convex NAG

Assume:

$$
\mu I \preceq H \preceq L I
$$

Condition number:

$$
\kappa = \frac{L}{\mu}
$$

---

## Parameters

$$
\eta = \frac{1}{L}
$$

$$
\beta =
\frac{\sqrt{\kappa} - 1}
{\sqrt{\kappa} + 1}
$$

---

## Linear Convergence

$$
f(w_k) - f^*
\le
C \left(1 - \sqrt{\mu/L}\right)^k
$$

This gives $\sqrt{\kappa}$ improvement over Gradient Descent.

---

## Strengths

- Linear convergence  
- Spectrally optimal on quadratics  
- Clean theoretical guarantees  

---

## Weaknesses

- Requires exact knowledge of $\mu$ and $L$  
- Fragile under curvature drift  
- Sensitive to noise  
- Assumes global strong convexity  

---

# Unified Comparison

| Method | Convex Rate | Strong Convex Rate | Robustness | Tuning Sensitivity |
|--------|-------------|-------------------|------------|-------------------|
| GD | $O(1/k)$ | $(1-\mu/L)^k$ | High | Low |
| Polyak | $O(1/k)$ | $\sqrt{\kappa}$ improvement | Medium | High |
| Convex NAG | $O(1/k^2)$ | Not linear | Medium | Medium |
| Strong NAG | — | $(1-\sqrt{\mu/L})^k$ | Low | Very High |

---

# Core Insight

Acceleration methods reshape the spectrum of the iteration matrix.

- Gradient Descent: contraction depends on $\kappa$
- Heavy Ball: contraction depends on $\sqrt{\kappa}$
- Strong NAG: contraction depends on $\sqrt{\mu/L}$

Faster convergence is achieved by pushing eigenvalues closer to the unit-circle boundary.

However:

Speed increases as stability margin decreases.

---

# Practical Conclusions

1. For clean quadratic problems: Strongly convex NAG is optimal.  
2. For smooth convex problems: Convex NAG gives optimal $O(1/k^2)$.  
3. For unknown curvature or noisy gradients: Heavy Ball or adaptive methods are more robust.  
4. In deep learning (non-convex, noisy, time-varying curvature): theoretical NAG must be modified.

---

# Final Principle

Acceleration improves asymptotic rate by reducing damping.

Reduced damping improves speed but decreases robustness.

Optimization design is always a tradeoff between:

- Speed  
- Stability  
- Adaptivity  
- Noise robustness  